# 12 — Stacking con Meta-Learner

**Materia:** Laboratorio de Implementacion II · Universidad Austral · Abril 2026

**Autores:** Roxana Alberti · Sandra Sschicchi · Fernando Paganini · Baltazar Villanueva · Paula Calviello · Rosana Martinez

---

## Que hace este notebook?

Implementamos **Stacking con meta-learner** como capa adicional sobre el ensemble LGB+XGB+CatBoost del nb11.

### ¿Que es Stacking?

Stacking es una tecnica de ensemble de dos niveles:

| Nivel | Que hace | Modelos |
|---|---|---|
| **Nivel 0** | Modelos base — producen predicciones OOF | LGB, XGB, CatBoost |
| **Nivel 1** | Meta-learner — combina las predicciones OOF | Logistic Regression |

La clave es que el meta-learner **aprende como combinar** los modelos base, en lugar de usar pesos fijos (como el blend 30/45/25 del nb10-11).

### ¿Por que no hay data leakage?

Los modelos base se entrenan con **5-fold CV**. Las predicciones OOF son predicciones de cada muestra hechas por un modelo que **nunca la vio durante el entrenamiento**. El meta-learner se entrena sobre estas predicciones OOF limpias.

```
Fold 1: [modelo base entrenado en folds 2-5] → predice fold 1
Fold 2: [modelo base entrenado en folds 1,3-5] → predice fold 2
...etc...
→ 14,993 predicciones OOF limpias → meta-learner sin leakage
```

### Meta-features

Para cada muestra de entrenamiento tenemos **15 probabilidades** como features del meta-learner:
- `lgb_p0 ... lgb_p4` (5 probs de LGB)
- `xgb_p0 ... xgb_p4` (5 probs de XGB)
- `cb_p0  ... cb_p4`  (5 probs de CatBoost)

+ **Optimized Rounder** aplicado sobre la salida del meta-learner (igual que nb11)


## Seccion A: Imports y datos

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from scipy.optimize import minimize
from functools import partial
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import label_binarize
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path.cwd()
while not (BASE_DIR / 'input').exists() and BASE_DIR != BASE_DIR.parent:
    BASE_DIR = BASE_DIR.parent
print(f'BASE_DIR: {BASE_DIR}')

SEED = 42
train_raw = pd.read_csv(BASE_DIR / 'input/train/train.csv')
sent_df   = pd.read_csv(BASE_DIR / 'input/train_sentiment_features.csv')
meta_df   = pd.read_csv(BASE_DIR / 'input/train_metadata_features.csv')
train_raw['desc_length'] = train_raw['Description'].fillna('').apply(len)

df = (train_raw
      .merge(sent_df[['PetID','sentiment_score','sentiment_magnitude','n_sentences']], on='PetID', how='left')
      .merge(meta_df[['PetID','avg_label_score','n_labels','crop_confidence']], on='PetID', how='left')
      .fillna(0))

train, test = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['AdoptionSpeed'])
print(f'Train: {len(train)} | Test: {len(test)}')


## Seccion B: Feature Engineering v4

In [ ]:
def target_encode(train_df, test_df, col, target='AdoptionSpeed', smoothing=10):
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    stats['encoded'] = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
    return train_df[col].map(stats['encoded']).fillna(global_mean), test_df[col].map(stats['encoded']).fillna(global_mean)

def add_features_v4(df_):
    df_ = df_.copy()
    df_['HasPhoto']            = (df_['PhotoAmt'] > 0).astype(int)
    df_['HasVideo']            = (df_['VideoAmt'] > 0).astype(int)
    df_['IsFree']              = (df_['Fee'] == 0).astype(int)
    df_['AgeGroup']            = pd.cut(df_['Age'], bins=[-1,3,12,48,9999], labels=[0,1,2,3]).astype(int)
    df_['HealthScore']         = ((df_['Vaccinated']==1).astype(int) + (df_['Dewormed']==1).astype(int) + (df_['Sterilized']==1).astype(int))
    df_['IsPureBreed']         = (df_['Breed2'] == 0).astype(int)
    df_['PhotoPerAnimal']      = df_['PhotoAmt'] / df_['Quantity'].replace(0,1)
    df_['Age_x_PhotoAmt']      = df_['Age'] * df_['PhotoAmt']
    df_['IsPureBreed_x_Age']   = df_['IsPureBreed'] * df_['AgeGroup']
    df_['HealthScore_x_Photo'] = df_['HealthScore'] * df_['HasPhoto']
    df_['IsYoungAndFree']      = ((df_['AgeGroup'] <= 1) & (df_['IsFree'] == 1)).astype(int)
    df_['IsHealthyAndPhoto']   = ((df_['HealthScore'] == 3) & (df_['HasPhoto'] == 1)).astype(int)
    df_['FeePerAnimal']        = df_['Fee'] / df_['Quantity'].replace(0,1)
    return df_

def nlp_feats(df_):
    desc = df_['Description'].apply(lambda x: '' if (x == 0 or str(x).strip() == '') else str(x))
    df_['word_count']      = desc.apply(lambda x: len(x.split()))
    df_['unique_words']    = desc.apply(lambda x: len(set(x.lower().split())))
    df_['avg_word_len']    = desc.apply(lambda x: round(sum(len(w) for w in x.split()) / max(len(x.split()),1), 2))
    df_['uppercase_ratio'] = desc.apply(lambda x: round(sum(c.isupper() for c in x) / max(len(x),1), 4))
    df_['has_exclamation'] = desc.apply(lambda x: int('!' in x))
    return df_

train = train.copy(); test = test.copy()
train['Breed1_enc'], test['Breed1_enc'] = target_encode(train, test, 'Breed1')
train['State_enc'],  test['State_enc']  = target_encode(train, test, 'State')

rescuer_count = train.groupby('RescuerID').size().rename('rescuer_n_pets')
train['rescuer_n_pets'] = train['RescuerID'].map(rescuer_count).fillna(1)
test['rescuer_n_pets']  = test['RescuerID'].map(rescuer_count).fillna(1)

age_med_map = train.groupby(['Breed1','Type'])['Age'].median().to_dict()
global_age  = train['Age'].median()
for df_ in [train, test]:
    df_['age_median_bt'] = [age_med_map.get((b,t), global_age) for b,t in zip(df_['Breed1'], df_['Type'])]
    df_['age_rel_breed'] = df_['Age'] / (df_['age_median_bt'] + 1)

train = nlp_feats(train); test = nlp_feats(test)
train_fe = add_features_v4(train); test_fe = add_features_v4(test)

ALL_FEATURES = [
    'Type','Age','Breed1','Breed2','Gender','Color1','Color2','Color3',
    'MaturitySize','FurLength','Vaccinated','Dewormed','Sterilized',
    'Health','Quantity','Fee','State','VideoAmt','PhotoAmt',
    'HasPhoto','HasVideo','IsFree','AgeGroup','HealthScore','IsPureBreed','PhotoPerAnimal',
    'Age_x_PhotoAmt','IsPureBreed_x_Age','HealthScore_x_Photo','IsYoungAndFree','IsHealthyAndPhoto','FeePerAnimal',
    'sentiment_score','sentiment_magnitude','n_sentences','avg_label_score','n_labels','crop_confidence','desc_length',
    'Breed1_enc','State_enc',
    'rescuer_n_pets','age_rel_breed','word_count','unique_words','avg_word_len','uppercase_ratio','has_exclamation'
]

X_train = train_fe[ALL_FEATURES]; X_test = test_fe[ALL_FEATURES]
y_train = train_fe['AdoptionSpeed']; y_test = test_fe['AdoptionSpeed']
print(f'Features: {len(ALL_FEATURES)}')


## Seccion C: Entrenamiento base con OOF predictions

Mismos parametros que nb11 (mejor ensemble). Guardamos las predicciones OOF para usarlas como features del meta-learner.

⏳ *Esta seccion tarda ~15-20 minutos.*

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

lgb_oof  = np.zeros((len(X_train), 5))
xgb_oof  = np.zeros((len(X_train), 5))
cb_oof   = np.zeros((len(X_train), 5))
lgb_test = np.zeros((len(X_test),  5))
xgb_test = np.zeros((len(X_test),  5))
cb_test  = np.zeros((len(X_test),  5))

lgb_params = {
    'objective': 'multiclass', 'num_class': 5, 'verbosity': -1,
    'num_leaves': 51, 'lambda_l1': 0.10, 'lambda_l2': 7.58,
    'feature_fraction': 0.59, 'bagging_fraction': 0.98,
    'bagging_freq': 1, 'min_child_samples': 118, 'learning_rate': 0.093
}
xgb_params = {
    'objective': 'multi:softprob', 'num_class': 5, 'verbosity': 0,
    'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 500,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.1,
    'reg_lambda': 1.0, 'min_child_weight': 5, 'random_state': SEED,
    'tree_method': 'hist', 'device': 'cpu'
}
cb_params = {
    'loss_function': 'MultiClass', 'eval_metric': 'Accuracy',
    'random_seed': SEED, 'verbose': False, 'early_stopping_rounds': 20,
    'iterations': 800, 'depth': 6, 'learning_rate': 0.19071087606036805,
    'l2_leaf_reg': 19.723764128240514, 'bagging_temperature': 0.22376847772452582,
    'border_count': 196, 'min_data_in_leaf': 13
}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f'  Fold {fold+1}/5...', end=' ')
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    m_lgb = lgb.train(lgb_params, lgb.Dataset(X_tr, label=y_tr), num_boost_round=500,
                      valid_sets=[lgb.Dataset(X_val, label=y_val)],
                      callbacks=[lgb.early_stopping(20, verbose=False)])
    lgb_oof[val_idx] = m_lgb.predict(X_val)
    lgb_test += m_lgb.predict(X_test)

    m_xgb = xgb.XGBClassifier(**xgb_params)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    xgb_oof[val_idx] = m_xgb.predict_proba(X_val)
    xgb_test += m_xgb.predict_proba(X_test)

    m_cb = CatBoostClassifier(**cb_params)
    m_cb.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
    cb_oof[val_idx] = m_cb.predict_proba(X_val)
    cb_test += m_cb.predict_proba(X_test)
    print('OK')

def norm(arr): return arr / arr.sum(axis=1, keepdims=True)
lgb_oof_n  = norm(lgb_oof);  xgb_oof_n  = norm(xgb_oof);  cb_oof_n  = norm(cb_oof)
lgb_test_n = norm(lgb_test); xgb_test_n = norm(xgb_test); cb_test_n = norm(cb_test)

# Blend de referencia (nb11)
W_LGB, W_XGB, W_CB = 0.30, 0.45, 0.25
oof_blend  = W_LGB * lgb_oof_n  + W_XGB * xgb_oof_n  + W_CB * cb_oof_n
test_blend = W_LGB * lgb_test_n + W_XGB * xgb_test_n + W_CB * cb_test_n

kappa_blend_test = cohen_kappa_score(y_test, test_blend.argmax(axis=1), weights='quadratic')
print(f'\nBlend (referencia nb10): Test kappa = {kappa_blend_test:.4f}')


## Seccion D: Meta-Learner (Logistic Regression)

Construimos la **segunda capa** del stacking con un meta-learner de Regresion Logistica.

### Meta-features (15 columnas por muestra)

```
[ lgb_p0, lgb_p1, lgb_p2, lgb_p3, lgb_p4,
  xgb_p0, xgb_p1, xgb_p2, xgb_p3, xgb_p4,
  cb_p0,  cb_p1,  cb_p2,  cb_p3,  cb_p4  ]
```

El meta-learner aprende **como ponderar** las probabilidades de cada modelo base para cada clase. A diferencia del blend fijo (30/45/25), puede aprender pesos **distintos por clase**.

### ¿Por que Logistic Regression?

- Simple y rapida de entrenar (15 features, ~15k muestras)
- No sobreajusta facilmente
- Produce probabilidades bien calibradas
- Interpretable: los coeficientes muestran que modelo confio mas para cada clase


In [ ]:
# Meta-features: concatenar las 15 probabilidades OOF
meta_X_train = np.hstack([lgb_oof_n, xgb_oof_n, cb_oof_n])  # shape: (n_train, 15)
meta_X_test  = np.hstack([lgb_test_n, xgb_test_n, cb_test_n]) # shape: (n_test, 15)
print(f'Meta-features shape: train={meta_X_train.shape}  test={meta_X_test.shape}')

# Meta-learner: Logistic Regression multinomial
meta_lr = LogisticRegression(
    multi_class='multinomial', solver='lbfgs',
    C=1.0, max_iter=1000, random_state=SEED
)
meta_lr.fit(meta_X_train, y_train)

# Predicciones del meta-learner
meta_oof_proba  = meta_lr.predict_proba(meta_X_train)  # para calibrar el rounder
meta_test_proba = meta_lr.predict_proba(meta_X_test)

kappa_meta_test = cohen_kappa_score(y_test, meta_lr.predict(meta_X_test), weights='quadratic')
print(f'Meta-Learner (LR) — Test kappa (argmax): {kappa_meta_test:.4f}')

# Mostrar coeficientes del meta-learner para cada clase
print('\nCoeficientes del meta-learner por clase (LGB / XGB / CB):')
feat_names = [f'lgb_p{i}' for i in range(5)] + [f'xgb_p{i}' for i in range(5)] + [f'cb_p{i}' for i in range(5)]
coef_df = pd.DataFrame(meta_lr.coef_, columns=feat_names, index=[f'clase_{i}' for i in range(5)])
print(coef_df.round(3).to_string())


## Seccion E: Optimized Rounder sobre la salida del meta-learner

Aplicamos el mismo Optimized Rounder del nb11, pero ahora sobre las probabilidades del **meta-learner**.

Score continuo: `score = meta_proba @ [0, 1, 2, 3, 4]`

Umbrales optimizados sobre las predicciones del meta-learner en los datos de entrenamiento (OOF clean).

In [ ]:
CLASS_WEIGHTS = np.array([0, 1, 2, 3, 4])
meta_oof_score  = meta_oof_proba  @ CLASS_WEIGHTS
meta_test_score = meta_test_proba @ CLASS_WEIGHTS

def apply_thresholds(score, thresholds):
    t = np.sort(thresholds)
    return np.clip(np.digitize(score, bins=t), 0, 4)

def kappa_loss(thresholds, score, y):
    preds = apply_thresholds(score, thresholds)
    return -cohen_kappa_score(y, preds, weights='quadratic')

init_thresholds = [0.5, 1.5, 2.5, 3.5]
print('Optimizando umbrales sobre predicciones del meta-learner...')
loss_fn = partial(kappa_loss, score=meta_oof_score, y=y_train.values)
result = minimize(loss_fn, init_thresholds, method='nelder-mead',
                  options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 20000})
optimal_thresholds = np.sort(result.x)

print(f'Umbrales optimos:  {[f"{t:.4f}" for t in optimal_thresholds]}')

# Kappa con Optimized Rounder
meta_test_preds_opt = apply_thresholds(meta_test_score, optimal_thresholds)
kappa_meta_opt = cohen_kappa_score(y_test, meta_test_preds_opt, weights='quadratic')

print(f'\nMeta-LR + Opt. Rounder — Test kappa: {kappa_meta_opt:.4f}')

# Referencia: nb11 con Optimized Rounder sobre blend
oof_blend_score  = oof_blend  @ CLASS_WEIGHTS
test_blend_score = test_blend @ CLASS_WEIGHTS
loss_fn_blend = partial(kappa_loss, score=oof_blend_score, y=y_train.values)
res_blend = minimize(loss_fn_blend, init_thresholds, method='nelder-mead',
                     options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 20000})
opt_t_blend = np.sort(res_blend.x)
kappa_nb11 = cohen_kappa_score(y_test, apply_thresholds(test_blend_score, opt_t_blend), weights='quadratic')
print(f'Blend + Opt. Rounder (nb11 ref) — Test kappa: {kappa_nb11:.4f}')

delta = kappa_meta_opt - kappa_nb11
print(f'\nDelta Stacking vs nb11: {delta:+.4f}')
if delta > 0:
    print(f'  -> El meta-learner MEJORA el blend fijo en {delta:+.4f}')
else:
    print(f'  -> El meta-learner no mejora el blend en este split ({delta:+.4f})')
    print(f'     El blend fijo (30/45/25) ya esta bien calibrado.')


## Seccion F: Comparativa final

In [ ]:
print('='*75)
print('  COMPARATIVA FINAL — Evolucion completa del proyecto')
print('='*75)
print(f'  Baseline (19 feat)                       Test: 0.3133')
print(f'  FE v1 (26 feat)                          Test: 0.3231  (+0.0099)')
print(f'  FE v2 + Optuna simple (32 feat)          Test: 0.3371  (+0.0238)')
print(f'  FE v3 + Optuna simple (39 feat)          Test: 0.3595  (+0.0462)')
print(f'  FE v4 + 5-fold CV (48 feat)              Test: 0.3867  (+0.0734)')
print(f'  Ensemble LGB+XGB+CB blend (nb10)         Test: 0.3951  (+0.0818)')
print(f'  Ensemble + Optimized Rounder (nb11)      Test: {kappa_nb11:.4f}  ({kappa_nb11-0.3133:+.4f} vs baseline)')
print(f'  Stacking LR + Opt. Rounder (nb12)        Test: {kappa_meta_opt:.4f}  ({kappa_meta_opt-0.3133:+.4f} vs baseline)')
print('='*75)
best = max(kappa_nb11, kappa_meta_opt)
print(f'  Mejor kappa del proyecto: {best:.4f}')
print('='*75)

# Distribucion de clases
from collections import Counter
print(f'\nDistribucion de predicciones Stacking + Opt. Rounder:')
dist_stack = Counter(meta_test_preds_opt)
dist_real  = Counter(y_test)
print(f'  {"Clase":<8} {"Real":<8} {"Stacking":<10}')
for cls in range(5):
    print(f'  {cls:<8} {dist_real.get(cls,0):<8} {dist_stack.get(cls,0):<10}')
